# Coefficient B09: validation

NS `i=N-1, j=N`. Synthetic endpoint fixtures isolate the mathematical reconstruction and do not require CCL or external survey files. The numerical source integrals use the original moving limits; they do not reuse the coefficient closed form or its series.

In [ ]:
import os
from pathlib import Path

import numpy

from limbercloud import ProjectPaths
from limbercloud.projection import jax_backend, numba_backend

# Retain the standard path object; these synthetic fixtures read no runtime data.
PATHS = ProjectPaths.from_root(os.environ.get('LIMBERCLOUD_RUNTIME_ROOT', Path.cwd()))


# Numerical integral

In [ ]:
def local_reference(family, number, left, right, power_left, power_right, z_left, z_right):
    """Integrate original source hats in dimensionless moving coordinates."""
    from scipy.integrate import quad
    width = right - left

    def source(r, rising):

        def integrand(v):
            hat = r + (1 - r) * v if rising else (1 - r) * (1 - v)
            u = left + width * (r + (1 - r) * v)
            return hat * v / u
        return (1 - r) ** 2 * quad(integrand, 0, 1, epsabs=2e-12, epsrel=2e-12)[0]

    def integrand(r):
        chi = left + width * r
        power = power_right * (chi / right) ** 3 if left == 0 else power_left * (1 - r) + power_right * r
        rz = (1 + z_left) * (1 - r) + (1 + z_right) * r
        if family == 'NS':
            density = r if number in (2, 10) else 1 - r
            return power * rz * density * source(r, number in (9, 10)) / chi
        first = source(r, number == 12)
        second = source(r, number != 1)
        return power * rz ** 2 * first * second
    exponent = 3 if family == 'NS' else 5
    return width ** exponent * quad(integrand, 0, 1, epsabs=2e-12, epsrel=2e-12)[0]

def integral_I9(chi1, chi2, power1, power2, redshift1, redshift2):
    return local_reference('NS', 9, chi1, chi2, power1, power2, redshift1, redshift2)


# Coefficient

The production function evaluates the preceding derivation with its stable analytic moments. Both compiled backends are compared to the independent integral. `p` is the normalized power parameter in the derivation; endpoint-linear evaluation is used when the right power vanishes.

In [ ]:
def coefficient_J9(chi1, chi2, power1, power2, redshift1, redshift2, backend=numba_backend):
    return float(numpy.asarray(backend.NS.element9(chi1, chi2, numpy.array([power1], dtype=numpy.float64), numpy.array([power2], dtype=numpy.float64), redshift1, redshift2))[0])


def compare(CHI1, CHI2, POWER1, POWER2, REDSHIFT1, REDSHIFT2):
    INTEGRAL = integral_I9(CHI1, CHI2, POWER1, POWER2, REDSHIFT1, REDSHIFT2)
    for BACKEND in (numba_backend, jax_backend):
        COEFFICIENT = coefficient_J9(CHI1, CHI2, POWER1, POWER2, REDSHIFT1, REDSHIFT2, BACKEND)
        ABSOLUTE_ERROR = abs(COEFFICIENT-INTEGRAL)
        RELATIVE_ERROR = COEFFICIENT/INTEGRAL-1 if INTEGRAL else None
        print(BACKEND.__name__, INTEGRAL, COEFFICIENT, RELATIVE_ERROR, ABSOLUTE_ERROR)
        numpy.testing.assert_allclose(COEFFICIENT, INTEGRAL, rtol=3e-10, atol=1e-60)


# Case 1: $\chi_1>0$

The evaluation interval is the actual final source interval; it is not a midpoint interval below the source support.

In [ ]:
CHI1, CHI2 = 0.7, 1.3
POWER1, POWER2 = 0.8, 1.1
REDSHIFT1, REDSHIFT2 = 0.2, 0.5
compare(CHI1, CHI2, POWER1, POWER2, REDSHIFT1, REDSHIFT2)


# Case 2: $\chi_1=0$

A single observer interval is also the final interval. Its power is cubic, not linear.

In [ ]:
CHI1, CHI2 = 0.0, 1.0
compare(CHI1, CHI2, 0.0, 1.0, 0.0, 0.0)


# Narrow intervals, signed and zero endpoint powers

In [ ]:
for CHI1, CHI2 in ((0.249,1.0), (0.25,1.0), (1.0,1.00001), (1.0,1.0000001)):
    for POWER1, POWER2 in ((0.8,1.1), (-0.4,0.7), (1.0,0.0), (0.0,0.0)):
        compare(CHI1, CHI2, POWER1, POWER2, 0.2, 0.5)
